In [ ]:
import paho.mqtt.client as mqtt 
import threading


topic_list = set()

def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("MQTT 연결 성공")
        client.subscribe("#")  # 모든 토픽 구독
    else:
        print(f"MQTT 연결 실패: {rc}")
        
def on_message(client, userdata, msg):
    print(f"Received message '{msg.payload.decode()}' on topic '{msg.topic}'")
    
def on_message(client, userdata, msg):
    topic_list.add(msg.topic)
    payload = msg.payload.decode("utf-8")
    msg_queue.put((msg.topic, payload, time.strftime('%Y-%m-%d %H:%M:%S')))

def start_mqtt():
    client = mqtt.Client()
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect("223.130.131.234", 31883, 60)  # 브로커 주소 및 포트
    client.loop_forever()

# --- MQTT 리스너 스레드 시작 ---
mqtt_thread = threading.Thread(target=start_mqtt)
mqtt_thread.daemon = True
mqtt_thread.start()



In [ ]:
import paho.mqtt.client as mqtt
import time
import threading
from collections import defaultdict

topic_stats = defaultdict(lambda: {"count": 0, "last_message": "", "last_timestamp": ""})

def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("MQTT 연결 성공")
        client.subscribe("#")  # 모든 토픽 구독
    else:
        print(f"MQTT 연결 실패: {rc}")
        
def on_message(client, userdata, msg):
    topic = msg.topic
    payload = msg.payload.decode("utf-8")
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S')

    # 통계 업데이트
    topic_stats[topic]["count"] += 1
    topic_stats[topic]["last_message"] = payload
    topic_stats[topic]["last_timestamp"] = timestamp

def start_mqtt():
    client = mqtt.Client()
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect("223.130.131.234", 31883, 60)
    client.loop_forever()

def print_stats_loop(interval=1):
    while True:
        print("\n--- MQTT 토픽 통계 ---")
        print(f"현재 시간: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        if topic_stats:
            for topic, stats in topic_stats.items():
                print(f"토픽: {topic}")
                print(f"  메시지 수: {stats['count']}")
                print(f"  마지막 메시지: {stats['last_message']}")
                print(f"  마지막 타임스탬프: {stats['last_timestamp']}")
            print("-------------------------\n")
        else:
            print("구독 중인 토픽이 없습니다.")
        time.sleep(interval)

if __name__ == "__main__":
    mqtt_thread = threading.Thread(target=start_mqtt, daemon=True)
    mqtt_thread.start()
    print_stats_loop(1)  # 통계 출력 루프 시작

In [ ]:
import matplotlib.pyplot as plt

plt.ion()  # 인터랙티브 모드 활성화

In [ ]:
import paho.mqtt.client as mqtt
import time
import threading
import matplotlib.pyplot as plt
from collections import defaultdict

topic_stats = defaultdict(lambda: {"count": 0, "last_message": "", "last_timestamp": ""})

def start_mqtt():
    client = mqtt.Client()
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect("223.130.131.234", 31883, 60)
    client.loop_forever()
    
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("MQTT 연결 성공")
        client.subscribe("#")  # 모든 토픽 구독
    else:
        print(f"MQTT 연결 실패: {rc}")

def on_message(client, userdata, msg):
    topic = msg.topic
    payload = msg.payload.decode("utf-8")
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S')

    # 통계 업데이트
    topic_stats[topic]["count"] += 1
    topic_stats[topic]["last_message"] = payload
    topic_stats[topic]["last_timestamp"] = timestamp

def plot_data():
    while True:
        if topic_stats:
            topics = list(topic_stats.keys())
            counts = [stats["count"] for stats in topic_stats.values()]

            plt.clf()  # 이전 플롯 지우기
            plt.bar(topics, counts)
            plt.xlabel('토픽')
            plt.ylabel('메시지 수')
            plt.title('MQTT 토픽 메시지 수')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.pause(1)  # 1초 대기 후 다시 그리기
        else:
            plt.clf()
            plt.text(0.5, 0.5, '구독 중인 토픽이 없습니다.', horizontalalignment='center', verticalalignment='center')
            plt.pause(1)    
if __name__ == "__main__":
    mqtt_thread = threading.Thread(target=start_mqtt, daemon=True)
    mqtt_thread.start()
    
    # 통계 출력 루프 시작
    stats_thread = threading.Thread(target=plot_data, daemon=True)
    stats_thread.start()

    
    plt.show()  # 플롯 창 표시

In [ ]:
import streamlit as st
import pandas as pd
import time
 
st.title('실시간 IoT 센서 데이터')
 
# 데이터 불러오기
data = pd.read_csv('_VEHICLE_ING_INFO_.csv')
 
# 데이터 실시간 표시
for i in range(len(data)):
    # 현재 데이터 포인트 표시
    st.line_chart(data[:i+1])
    
    # 다음 데이터 포인트를 표시하기 전에 잠시 대기
    time.sleep(0.1)

In [ ]:
import pygwalker as pyg
import pandas as pd

df = pd.read_csv('_VEHICLE_ING_INFO_.csv')
pyg.walk(df, server_mode=True, port=8080)  # 서버 모드로 실행

In [ ]:
import pygwalker as pyg
import pandas as pd
import paho.mqtt.client as mqtt
import time
import threading
from collections import defaultdict
topic_stats = defaultdict(lambda: {"count": 0, "last_message": "", "last_timestamp": ""})
def start_mqtt():
    client = mqtt.Client()
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect("223.130.131.234", 31883, 60)
    client.loop_forever()
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("MQTT 연결 성공")
        client.subscribe("#")  # 모든 토픽 구독
    else:
        print(f"MQTT 연결 실패: {rc}")
def on_message(client, userdata, msg):
    topic = msg.topic
    payload = msg.payload.decode("utf-8")
    timestamp = time.strftime('%Y-%m-%d %H:%M:%S')

    # 통계 업데이트
    topic_stats[topic]["count"] += 1
    topic_stats[topic]["last_message"] = payload
    topic_stats[topic]["last_timestamp"] = timestamp
    
def plot_data():
    while True:
        if topic_stats:
            topics = list(topic_stats.keys())
            counts = [stats["count"] for stats in topic_stats.values()]

            df = pd.DataFrame({
                '토픽': topics,
                '메시지 수': counts
            })
            
            pyg.walk(df, server_mode=True, port=8090)  # 서버 모드로 실행
            time.sleep(1)  # 1초 대기 후 다시 그리기
        else:
            print("구독 중인 토픽이 없습니다.")
            time.sleep(1)  # 잠시 대기 후 다시 확인
if __name__ == "__main__":
    mqtt_thread = threading.Thread(target=start_mqtt, daemon=True)
    mqtt_thread.start()
    
    # 통계 출력 루프 시작
    stats_thread = threading.Thread(target=plot_data, daemon=True)
    stats_thread.start()
    

Box(children=(HTML(value='\n<div id="ifr-pyg-000636b4f167718axIWpFGYeQTMHA4Ni" style="height: auto">\n    <hea…